In [1]:
from pathlib import Path

from sentence_transformers import SentenceTransformer
from config import config
import random, numpy as np, pandas as pd, os
from transformers import AutoTokenizer, AutoModel
import torch

from dotenv import load_dotenv

d:\MSc\3. Spring 2026\CSE715\Project\vae-audio-clustering\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv() 
token = os.environ.get("HF_TOKEN", None)
random.seed(42)

In [3]:
root = Path("../..")

In [4]:
LYRICS_DIR_EN = root / config.LYRICS_DIR_EN
LYRICS_DIR_BN = root / config.LYRICS_DIR_BN
EMBEDDINGS_DIR = root / config.EMBEDDINGS_DIR
METADATA_EN = root / config.METADATA_DIR / "metadata_en.csv"
METADATA_BN = root / config.METADATA_DIR / "bn" / "metadata_bn.csv"

In [6]:
LYRICS_DIR_EN.exists(), LYRICS_DIR_BN.exists(), METADATA_EN.exists(), METADATA_BN.exists()

(True, True, True, True)

In [7]:
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
model = SentenceTransformer('all-mpnet-base-v2', token=token)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13048.19it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
bn_tokenizer = AutoTokenizer.from_pretrained("csebuetnlp/banglabert", token=token)
bn_model = AutoModel.from_pretrained("csebuetnlp/banglabert", token=token)
bn_model.eval()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 31961.86it/s]
ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
electra.embeddings.position_ids                   | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ElectraModel(
  (embeddings): ElectraEmbeddings(
    (word_embeddings): Embedding(32000, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): ElectraEncoder(
    (layer): ModuleList(
      (0-11): 12 x ElectraLayer(
        (attention): ElectraAttention(
          (self): ElectraSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): ElectraSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0

In [10]:
def encode_banglabert(text: str) -> np.ndarray:
    inputs = bn_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    )
    with torch.no_grad():
        outputs = bn_model(**inputs)
    # Mean-pool the last hidden state across tokens
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
    return embedding

In [11]:
lyric_stems = [
    p.stem for p in LYRICS_DIR_BN.iterdir()
    if p.is_file() and p.suffix.lower() == ".txt"
]

In [12]:
df_meta_bn = pd.read_csv(METADATA_BN)
relevant_meta_bn = df_meta_bn[df_meta_bn["id"].isin(lyric_stems)]

In [13]:
relevant_meta_bn.shape

(341, 5)

In [14]:
ok, skipped, errors = 0, 0, 0

for lyric_stem in lyric_stems:
    row = relevant_meta_bn[relevant_meta_bn["id"] == lyric_stem]

    if row.empty:
        print(f"  [SKIP] {lyric_stem} — not found in metadata")
        skipped += 1
        continue

    song_id = str(row["id"].item())
    out_path = EMBEDDINGS_DIR / f"{song_id}.npy"

    if out_path.exists():
        print(f"  [EXISTS] {song_id} — already embedded, skipping")
        skipped += 1
        continue

    try:
        lyric_path = LYRICS_DIR_BN / f"{lyric_stem}.txt"
        with open(lyric_path, "r", encoding="utf-8") as f:
            text = f.read()

        z_text = encode_banglabert(text)
        np.save(out_path, z_text)
        print(f"  [OK] {song_id}")
        ok += 1

    except Exception as e:
        print(f"  [ERROR] {song_id}: {e}")
        errors += 1

print(f"  → Done: {ok} saved, {skipped} skipped, {errors} errors")

  [OK] -Bfa-aVa7MU_seg_00001
  [OK] -Bfa-aVa7MU_seg_00002
  [OK] -Bfa-aVa7MU_seg_00003
  [OK] -Bfa-aVa7MU_seg_00004
  [OK] -Bfa-aVa7MU_seg_00005
  [OK] -Bfa-aVa7MU_seg_00006
  [OK] -Bfa-aVa7MU_seg_00007
  [OK] -Bfa-aVa7MU_seg_00008
  [OK] -Bfa-aVa7MU_seg_00009
  [OK] -ph4mykFp9I_seg_00001
  [OK] -ph4mykFp9I_seg_00002
  [OK] -ph4mykFp9I_seg_00003
  [OK] -ph4mykFp9I_seg_00004
  [OK] -ph4mykFp9I_seg_00005
  [OK] -ph4mykFp9I_seg_00006
  [OK] -ph4mykFp9I_seg_00007
  [OK] -ph4mykFp9I_seg_00008
  [OK] -ph4mykFp9I_seg_00009
  [OK] -ph4mykFp9I_seg_00010
  [OK] -ph4mykFp9I_seg_00011
  [OK] -ph4mykFp9I_seg_00012
  [OK] -ph4mykFp9I_seg_00013
  [OK] -ph4mykFp9I_seg_00014
  [OK] -ph4mykFp9I_seg_00015
  [OK] -ph4mykFp9I_seg_00016
  [OK] 8dLOs8LK4OQ_seg_00001
  [OK] 8dLOs8LK4OQ_seg_00002
  [OK] 8dLOs8LK4OQ_seg_00003
  [OK] 8dLOs8LK4OQ_seg_00004
  [OK] 8dLOs8LK4OQ_seg_00005
  [OK] 8dLOs8LK4OQ_seg_00006
  [OK] 8dLOs8LK4OQ_seg_00007
  [OK] 8dLOs8LK4OQ_seg_00008
  [OK] 8dLOs8LK4OQ_seg_00009
  [OK] 8dLOs8L

In [15]:
en_lyric_list = [p.stem for p in LYRICS_DIR_EN.iterdir() if p.is_file() and p.suffix.lower() == ".txt"]

In [16]:
df_meta_en = pd.read_csv(METADATA_EN)

In [17]:
relevant_meta_en = df_meta_en[df_meta_en['lyric_file_stem'].isin(en_lyric_list)]

In [18]:
for lyric_path in en_lyric_list:
    song_id = relevant_meta_en[relevant_meta_en["lyric_file_stem"]==lyric_path]["audio_file_stem"].item()
    song_id = str(song_id)
    print(song_id)
    
    try:
        main_file_path = LYRICS_DIR_EN / f"{lyric_path}.txt"
        if main_file_path.exists():
            with open(main_file_path, 'r', encoding='utf-8') as f:
                text = f.read()
            
            # Generate vector (384 dimensions)
            z_text = model.encode(text)
            
            # Save as .npy using the song_id as name
            np.save(EMBEDDINGS_DIR / f"{song_id}.npy", z_text)
        else:
            print(f"Warning: File not found at {main_file_path}")
            
    except Exception as e:
        print(f"Error processing {song_id}: {e}")

A025
A039
A048
A167
MT0000027422
MT0000082188
MT0000144160
MT0000216849
MT0000277633
MT0000289116
MT0000315392
MT0000321394
MT0000421504
MT0000487439
MT0000688312
MT0000764951
MT0000766316
MT0001183783
MT0001265984
MT0001297031
MT0001347462
MT0001353828
MT0001356476
MT0001356637
MT0001385947
MT0001514284
MT0001566152
MT0001729778
MT0002024741
MT0002068648
MT0002277261
MT0002336535
MT0002541988
MT0002645049
MT0002737189
MT0002859848
MT0002962258
MT0003343422
MT0003510796
MT0003629164
MT0003859252
MT0004033010
MT0004085907
MT0004240748
MT0004263301
MT0004293364
MT0004338270
MT0004362501
MT0004395306
MT0004683099
MT0004725405
MT0004776609
MT0004781165
MT0005083885
MT0005115042
MT0005404247
MT0005540907
MT0005625762
MT0005630273
MT0005959692
MT0006137372
MT0006357849
MT0006421356
MT0006708402
MT0006806145
MT0006983241
MT0007058512
MT0007508173
MT0007663424
MT0007868583
MT0008361425
MT0008537015
MT0008554680
MT0008632122
MT0008684978
MT0008966305
MT0008972801
MT0008999898
MT0009153900
MT000